In [1]:
!python -V

Python 3.10.6


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [6]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1717092736403, experiment_id='1', last_update_time=1717092736403, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [7]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [8]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

In [9]:
df_train.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'duration'],
      dtype='object')

In [10]:
len(df_train), len(df_val)

(73908, 61921)

In [11]:
df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)

df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)
df_val['PU_DO'] = df_val['PULocationID'].astype(str) + '_' + df_val['DOLocationID'].astype(str)

In [12]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [13]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [14]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.758715208946364

In [15]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [16]:
with mlflow.start_run():

    mlflow.set_tag("developer", "rui pinto")

    mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")

    alpha = 0.01
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [17]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

# objective function
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=10
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

# search space
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

# optimization
best_result = fmin(
    fn=objective, # objective function
    space=search_space, # search space
    algo=tpe.suggest, # surrogate algorithm
    max_evals=2, # number of iterations
    trials=Trials() # trials object that keeps track of the sample results
)

  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [16:06:43] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.68800                         
[1]	validation-rmse:9.55244                          
[2]	validation-rmse:8.71783                          
[3]	validation-rmse:8.10294                          
[4]	validation-rmse:7.66291                          
[5]	validation-rmse:7.36018                          
[6]	validation-rmse:7.13952                          
[7]	validation-rmse:6.97838                          
[8]	validation-rmse:6.87206                          
[9]	validation-rmse:6.79251                          
[10]	validation-rmse:6.72963                         
[11]	validation-rmse:6.68843                         
[12]	validation-rmse:6.65231                         
[13]	validation-rmse:6.62426                         
[14]	validation-rmse:6.60345                         
[15]	validation-rmse:6.58639                         
[16]	validation-rmse:6.57381                         
[17]	validation-rmse:6.56384                         
[18]	validation-rmse:6.55415

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [16:07:08] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.29891                                                   
[1]	validation-rmse:7.85996                                                   
[2]	validation-rmse:7.19391                                                   
[3]	validation-rmse:6.89148                                                   
[4]	validation-rmse:6.74481                                                   
[5]	validation-rmse:6.66850                                                   
[6]	validation-rmse:6.62426                                                   
[7]	validation-rmse:6.59473                                                   
[8]	validation-rmse:6.58203                                                   
[9]	validation-rmse:6.57239                                                   
[10]	validation-rmse:6.56668                                                  
[11]	validation-rmse:6.56307                                                  
[12]	validation-rmse:6.55854                        

In [18]:
best_result

{'learning_rate': 0.3886379233814974,
 'max_depth': 23.0,
 'min_child_weight': 1.0017358151177247,
 'reg_alpha': 0.2323207188900079,
 'reg_lambda': 0.05832051700732755}

In [19]:
# disable autolog
mlflow.xgboost.autolog(disable=True)

In [20]:
with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.06549179851966823,
        'max_depth': 65,
        'min_child_weight': 4.730482762129463,
        'objective': 'reg:linear',
        'reg_alpha': 0.009451536770262137,
        'reg_lambda': 0.14515981980661224,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=10
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [16:07:36] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:11.68345
[1]	validation-rmse:11.19827
[2]	validation-rmse:10.75456
[3]	validation-rmse:10.34939
[4]	validation-rmse:9.98008
[5]	validation-rmse:9.64399
[6]	validation-rmse:9.33860
[7]	validation-rmse:9.06146
[8]	validation-rmse:8.81024
[9]	validation-rmse:8.58308
[10]	validation-rmse:8.37812
[11]	validation-rmse:8.19259
[12]	validation-rmse:8.02590
[13]	validation-rmse:7.87547
[14]	validation-rmse:7.74046
[15]	validation-rmse:7.61856
[16]	validation-rmse:7.50947
[17]	validation-rmse:7.41096
[18]	validation-rmse:7.32291
[19]	validation-rmse:7.24386
[20]	validation-rmse:7.17312
[21]	validation-rmse:7.10941
[22]	validation-rmse:7.05222
[23]	validation-rmse:7.00106
[24]	validation-rmse:6.95501
[25]	validation-rmse:6.91375
[26]	validation-rmse:6.87678
[27]	validation-rmse:6.84261
[28]	validation-rmse:6.81242
[29]	validation-rmse:6.78495
[30]	validation-rmse:6.76033
[31]	validation-rmse:6.73791
[32]	validation-rmse:6.71790
[33]	validation-rmse:6.69978
[34]	validation-rmse

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [16:08:41] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)


In [21]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog(disable=True)

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
        mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)


/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/sklearn/svm/_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
